## How to run this notebook

**What it does.** Tests whether an unsupervised Isolation Forest can find account-opening fraud in BAF Base (task A2-13), and compares it with the team's earlier supervised logistic regression on PR-AUC and recall at 5% false-positive rate. It tunes the forest on time-ordered folds, picks the contamination setting, scores month 5 and draws `fig12` to `fig14` (precision–recall, anomaly score by class with recall as FPR rises to 20%, and precision/recall/F1 against contamination). The stored cell outputs and figures are kept in this file.

**Data used.** Development months 0–5 only. Months 6–7 are dropped when the data are loaded. The forest is tuned on folds that train on months 0–2 and 0–3 and validate on months 3 and 4, then trained on months 0–4 and scored on all 119,323 applications of month 5.

**Where to put the data.** The raw file is not in Git. Download `Base.csv` (or the Kaggle `Base.csv.zip`) from <https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022> and place it in `experiment_suite/dataset/baf/`. The setup cell also accepts `notebooks/data/` or the repository root, and reads the zip directly.

**To run.** Use the repository's conda environment (`conda activate fraudsuite`, see the README) with Jupyter installed (`pip install jupyterlab` if missing). Open this file and choose Run All (about 7 minutes when it was checked). Paths are relative, so it works from `notebooks/` or from the repository root. The helper code is in `notebooks/a2_analysis/isoforest.py` (and `eda.py` for the shared loading and plotting helpers). The seed is fixed (30049).

**Files it writes** (all under `notebooks/a2_outputs/`, safe to delete and re-create). This notebook saves its figures to files instead of showing them inline, so the three figures and the result tables from the checked run are also kept in the repository at these paths:
- `figures/fig12_iforest_precision_recall.png`, `fig13_iforest_score_and_recall_fpr.png`, `fig14_iforest_contamination.png`.
- `tables/isoforest_*.csv`: the tuning grid, the contamination table, the seed spread, the comparison with the logistic regression and the supporting overlap and KS tables.

# 03 – Isolation Forest anomaly detection vs supervised scores

**COS30049 Assignment 2 · Group COS30049.2_1 · Task A2-13 (HD: additional ML)**
Owner: Thông · Reviewer: Phúc

**Goal.** Test whether an *unsupervised* Isolation Forest can find account-opening fraud in BAF Base, and compare it with supervised scores on the metrics the unit cares about: PR-AUC and recall at 5% false-positive rate (FPR).

**Scope:** contamination tuning and historical Isolation Forest versus logistic-regression diagnostics; this is not a comparison with the current v4 CatBoost champion.

**Rules followed (from the project instructions).**
- This study loads development months 0–5 only. Other repository experiments have already inspected months 6–7; they are not a fresh blind holdout.
- Every preprocessing step is fitted on the training months of each fold only.
- Metrics reported: PR-AUC, recall and precision at 5% FPR, F1. Accuracy is not used.
- The current production fairness audit uses age >=50. Historical comparisons of both definitions in the EDA do not change that setting.

**Interpretation:** ROC-derived recall at 5% FPR uses validation labels retrospectively. It is not an independently calibrated deployment threshold. See `notebooks/README.md` for the historical baseline and current-pipeline distinctions.

**Comparison model.** The supervised model is the team's A2-11 classifier from Kha's repo (`KennyDao1324/ai4cyber-fraud`, commit 661871e): same preprocessing (including -1 to NaN in the six sentinel columns), same 26 input features, same class-balanced logistic regression. The repo's frozen artifact is not used. Instead `isoforest.fit_team_model` rebuilds the same pipeline and re-fits it on each time-ordered fold, so it never sees months 6–7 (the repo trains on months 0–4, validates on month 5 and tests on months 6–7). On the same fold the rebuilt pipeline matches the repo's own code to within 1e-7 on every score.

## 1. Setup

In [1]:
# Imports and project paths
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ROOT = the notebooks/ folder, found by walking up from the working directory until a2_analysis/eda.py appears.
_ROOTS = [p for base in [Path.cwd(), *Path.cwd().parents] for p in (base, base / "notebooks")]
ROOT = next((p for p in _ROOTS if (p / "a2_analysis" / "eda.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the repository (a2_analysis/eda.py not found).")
sys.path.insert(0, str(ROOT))
REPO = ROOT.parent

from a2_analysis import eda, isoforest as iso

# Raw BAF Base (Kaggle v2); never committed. Unzipped CSV first, then the Kaggle zip.
RAW = REPO / "experiment_suite" / "dataset" / "baf"
CANDIDATES = [RAW / "Base.csv", RAW / "Base.csv.zip", ROOT / "data" / "Base.csv", ROOT / "data" / "Base.csv.zip",
              REPO / "Base.csv", REPO / "Base.csv.zip"]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Base.csv not found: put Base.csv or the Kaggle Base.csv.zip in "
                            + ", ".join(os.path.relpath(p, REPO) for p in CANDIDATES))

# Everything this notebook writes goes under notebooks/a2_outputs/.
OUT_DIR = ROOT / "a2_outputs"
FIG_DIR, TABLE_DIR = OUT_DIR / "figures", OUT_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
eda.set_style()
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("Data:", os.path.relpath(DATA_PATH, REPO), "| seed:", iso.SEED)

Data: experiment_suite/dataset/baf/Base.csv.zip | seed: 30049


## 2. Load the development months

In [2]:
# Months 6-7 are dropped inside load_dev; missing codes (-1 / negative) become NaN
dev = iso.load_dev(DATA_PATH)
assert set(dev["month"].unique()) == set(eda.DEV_MONTHS), "only months 0-5 may be used here"
print(f"Development months 0-5: {len(dev):,} rows, {dev[eda.TARGET].sum():,} frauds, "
      f"fraud rate {dev[eda.TARGET].mean():.4%}")
numeric, flags, categorical = iso.feature_columns(dev)
print(f"Inputs: {len(numeric)} numeric + {len(flags)} flags + {len(categorical)} categorical = "
      f"{len(numeric) + len(flags) + len(categorical)}")
print("Excluded: month, device_fraud_count (constant) and the protected features", iso.PROTECTED_FEATURES)

# Same rows with the raw BAF missing codes kept: the team pipeline turns -1 into NaN itself
dev_raw = iso.load_dev(DATA_PATH, missing_as_nan=False)
assert dev_raw.index.equals(dev.index)

Development months 0-5: 794,989 rows, 8,151 frauds, fraud rate 1.0253%


Inputs: 16 numeric + 6 flags + 4 categorical = 26
Excluded: month, device_fraud_count (constant) and the protected features ['customer_age', 'income', 'employment_status']


## 3. Validation design

The EDA (Finding 12) showed strong drift inside months 0–5, so a random split would be too optimistic. Validation is **time-ordered** (expanding window): train on every earlier month, score the next one.

| Fold | Train months | Validation month | Used for |
|---|---|---|---|
| A | 0–2 | 3 | choosing settings |
| B | 0–3 | 4 | choosing settings |
| C | 0–4 | 5 | **held back**: reported once, never tuned on |

Fold C is exactly the team repo's own train/validation split (train months 0–4, validate month 5), so the comparison is on the team's validation month. The Isolation Forest never sees labels when it is fitted. Labels are used only to score it.

In [3]:
rows = []
for val_month in (3, 4, 5):
    train, val = iso.split_by_month(dev, range(val_month), val_month)
    rows.append({"fold": "ABC"[val_month - 3], "train_months": f"0-{val_month - 1}", "val_month": val_month,
                 "train_rows": len(train), "val_rows": len(val), "val_frauds": int(val[eda.TARGET].sum()),
                 "val_fraud_rate_pct": 100 * val[eda.TARGET].mean()})
folds = pd.DataFrame(rows)
folds

,fold,train_months,val_month,train_rows,val_rows,val_frauds,val_fraud_rate_pct
0,A,0-2,3,397039,150936,1392,0.9222
1,B,0-3,4,547975,127691,1452,1.1371
2,C,0-4,5,675666,119323,1411,1.1825


## 4. Tune the forest settings

Grid: feature set (numeric only, or numeric plus the five categoricals one-hot encoded) × number of trees × `max_samples` × `max_features`. Missing values are median-filled and each gap gets an indicator column (missingness is a fraud signal, EDA Finding 3).

`contamination` is deliberately **not** in this grid. In scikit-learn it only decides where the yes/no cut-off falls; it never changes the anomaly scores, so PR-AUC and recall at 5% FPR cannot depend on it. It is tuned separately in Section 5.

In [4]:
# 16 settings x 2 tuning folds; takes a couple of minutes
grid = iso.tune_isolation_forest(dev)
grid.to_csv(TABLE_DIR / "isoforest_grid_folds.csv", index=False)
summary, best = iso.best_settings(grid)
summary.to_csv(TABLE_DIR / "isoforest_grid_summary.csv", index=False)
print("Best settings (mean over folds A and B, ranked by PR-AUC):")
print(best.to_dict())
summary.head(8)

Best settings (mean over folds A and B, ranked by PR-AUC):
{'feature_set': 'with_categorical', 'n_estimators': 300, 'max_samples': 256, 'max_features': 0.5, 'pr_auc': 0.015105948448999085, 'roc_auc': 0.6135232992812907, 'recall_at_5fpr': 0.09128016053956492}


,feature_set,n_estimators,max_samples,max_features,pr_auc,roc_auc,recall_at_5fpr
0,with_categorical,300,256,0.5000,0.0151,0.6135,0.0913
1,with_categorical,300,1024,0.5000,0.0146,0.6085,0.0883
2,with_categorical,100,256,0.5000,0.0144,0.5944,0.0859
3,with_categorical,300,1024,1.0000,0.0137,0.5931,0.0781
4,with_categorical,100,1024,0.5000,0.0137,0.5919,0.0792
5,with_categorical,300,256,1.0000,0.0133,0.5803,0.0776
6,with_categorical,100,256,1.0000,0.0129,0.5736,0.0722
7,numeric,300,1024,1.0000,0.0129,0.5623,0.0813


In [5]:
# How big is the gap between settings compared with plain seed-to-seed noise?
# (Refit the chosen settings with 5 seeds on the held-back month 5.)
spread = iso.seed_spread(dev, best)
spread.to_csv(TABLE_DIR / "isoforest_seed_spread.csv", index=False)
print(f"Grid, mean PR-AUC across settings: min {summary['pr_auc'].min():.4f}, max {summary['pr_auc'].max():.4f}")
print(f"Seeds, PR-AUC on month 5 for the chosen settings: min {spread['pr_auc'].min():.4f}, max {spread['pr_auc'].max():.4f}")
spread

Grid, mean PR-AUC across settings: min 0.0121, max 0.0151
Seeds, PR-AUC on month 5 for the chosen settings: min 0.0135, max 0.0150


,seed,pr_auc,roc_auc,recall_at_5fpr
0,30049,0.0150,0.5683,0.0787
1,1,0.0135,0.5440,0.0609
2,2,0.0137,0.5431,0.0581
3,3,0.0142,0.5609,0.0687
4,4,0.0140,0.5483,0.0737


## 5. Tune `contamination`

For the chosen settings, sweep the share of applications flagged. The cut-off is the matching quantile of the **training** anomaly scores; precision, recall and F1 are measured on the validation month. Folds A and B only.

Selection rule: the smallest `contamination` whose mean F1 is within 5% of the best. The forest is a weak ranker, so F1 keeps edging up as more applications are flagged and the raw maximum would sit on the edge of the grid; the start of the plateau is the more sensible operating point.

In [6]:
sweep = iso.tune_contamination(dev, best)
sweep.to_csv(TABLE_DIR / "isoforest_contamination.csv", index=False)
by_c = sweep.groupby("contamination")[["precision", "recall", "f1", "flag_rate", "pr_auc"]].mean()
CONTAMINATION = iso.choose_contamination(by_c["f1"])
print(f"Highest mean F1: {by_c['f1'].max():.4f} at contamination {by_c['f1'].idxmax()}")
print(f"Chosen contamination (start of the plateau, F1 {by_c.loc[CONTAMINATION, 'f1']:.4f}): {CONTAMINATION}")
by_c

Highest mean F1: 0.0323 at contamination 0.1
Chosen contamination (start of the plateau, F1 0.0311): 0.05


,precision,recall,f1,flag_rate,pr_auc
contamination,,,,,
0.0050,0.0246,0.0130,0.0170,0.0054,0.0151
0.0100,0.0220,0.0240,0.0228,0.0110,0.0151
0.0200,0.0188,0.0406,0.0256,0.0218,0.0151
0.0300,0.0188,0.0611,0.0287,0.0327,0.0151
0.0500,0.0185,0.0999,0.0311,0.0546,0.0151
0.0750,0.0179,0.1440,0.0317,0.0815,0.0151
0.1000,0.0177,0.1887,0.0323,0.1083,0.0151


In [7]:
# Confirm our cut-off logic reproduces scikit-learn's own `contamination` argument
mismatch = iso.check_matches_sklearn(dev, best, CONTAMINATION)
print(f"Share of month-5 rows where manual cut-off and sklearn predict() disagree: {mismatch:.4%}")
assert mismatch < 1e-3
iso_fig = eda.save_figure(iso.plot_contamination(sweep, CONTAMINATION), "fig14_iforest_contamination", FIG_DIR)
print(iso_fig.relative_to(ROOT))

Share of month-5 rows where manual cut-off and sklearn predict() disagree: 0.0000%
a2_outputs/figures/fig14_iforest_contamination.png


## 6. Compare with supervised scores

Same folds and validation months for both. The team logistic regression uses fraud labels and its own preprocessing (250,000-row stratified training sample, -1 turned into NaN only in its six sentinel columns); the forest uses no labels and the NaN-plus-indicator treatment from Section 4 (every BAF missing code, including negative `intended_balcon_amount`). Both see the same 26 input features. Fold C (month 5) is the clean comparison because nothing was tuned on it. For the F1 columns both models flag the same share of applications as the forest does at the tuned `contamination`.

In [8]:
comparison, scores = iso.compare_models(dev, dev_raw, best, CONTAMINATION)
comparison.to_csv(TABLE_DIR / "isoforest_comparison.csv", index=False)
cols = ["val_month", "model", "pr_auc", "roc_auc", "recall_at_5fpr", "precision_at_5fpr", "precision", "recall", "f1", "flag_rate"]
comparison[cols]

,val_month,model,pr_auc,roc_auc,recall_at_5fpr,precision_at_5fpr,precision,recall,f1,flag_rate
0,3,Isolation Forest,0.0147,0.6308,0.1013,0.0186,0.0187,0.1171,0.0322,0.0578
1,3,Logistic regression (team repo),0.1240,0.8950,0.5136,0.0876,0.0846,0.5302,0.1459,0.0578
2,4,Isolation Forest,0.0155,0.5963,0.0813,0.0188,0.0183,0.0826,0.0300,0.0514
3,4,Logistic regression (team repo),0.1331,0.8909,0.4979,0.1041,0.1071,0.4842,0.1755,0.0514
4,5,Isolation Forest,0.0150,0.5683,0.0787,0.0185,0.0196,0.0595,0.0295,0.0359
5,5,Logistic regression (team repo),0.1482,0.8751,0.4954,0.1064,0.1339,0.4061,0.2014,0.0359


In [9]:
# Held-back month 5: figures
no_skill = scores[5]["Isolation Forest"][0].mean()
print(f"Month 5 fraud rate (no-skill PR-AUC): {no_skill:.4f}")
for name, fig in [("fig12_iforest_precision_recall", iso.plot_pr_curves(scores[5])),
                  ("fig13_iforest_score_and_recall_fpr", iso.plot_score_and_roc(scores[5]))]:
    print(eda.save_figure(fig, name, FIG_DIR).relative_to(ROOT))

Month 5 fraud rate (no-skill PR-AUC): 0.0118


a2_outputs/figures/fig12_iforest_precision_recall.png
a2_outputs/figures/fig13_iforest_score_and_recall_fpr.png


## 7. Does the forest find anything the classifier misses?

At 5% FPR on month 5 each model raises an alarm on about 5% of legitimate applications. Which frauds does each catch, and do the two scores agree?

In [10]:
overlap, rho = iso.overlap_table(scores[5])
overlap.to_csv(TABLE_DIR / "isoforest_overlap.csv")
print({k: round(float(v), 3) for k, v in rho.items()})
overlap

{'spearman_all': -0.043, 'spearman_fraud_only': 0.269}


,frauds,share_of_frauds_pct
caught by both,74,5.2445
only Isolation Forest,37,2.6223
only team logistic regression,625,44.2948
caught by neither,675,47.8384


## 8. Diagnostic: is the forest weak because it sees too many features?

Per fold, keep only the top k numeric features by KS statistic **computed on the training months**, then refit. This uses fraud labels to choose features, so it is *semi*-supervised and not a pure anomaly detector. It only asks whether the poor result is caused by noise columns.

In [11]:
ks_sel = iso.ks_selected_forest(dev, best)
ks_sel.to_csv(TABLE_DIR / "isoforest_ks_selected.csv", index=False)
print("Features chosen for k = 4 in month 5:", ks_sel[(ks_sel.val_month == 5) & (ks_sel.top_k_features == 4)]["features"].iloc[0])
ks_sel.drop(columns="features")

Features chosen for k = 4 in month 5: current_address_months_count, credit_risk_score, proposed_credit_limit, date_of_birth_distinct_emails_4w


,val_month,top_k_features,pr_auc,roc_auc,recall_at_5fpr
0,3,4,0.0280,0.7088,0.1990
1,3,8,0.0148,0.6288,0.0991
2,4,4,0.0316,0.6989,0.1915
3,4,8,0.0183,0.6307,0.1047
4,5,4,0.0271,0.6451,0.1765
5,5,8,0.0168,0.5938,0.0879


## 9. Findings

**Finding 1 – the Isolation Forest is barely better than chance.**
On the held-back month 5 (train 0–4) it reaches PR-AUC 0.015 against a no-skill level of 0.012 (the fraud rate), ROC-AUC 0.57, and recall at 5% FPR of 7.9% with precision 1.9%. The team's logistic regression reaches PR-AUC 0.148, ROC-AUC 0.875, recall at 5% FPR of 49.5% and precision 10.6%: about 10 times better on PR-AUC and 6 times on recall. For context, the team repo reports PR-AUC 0.157 on its month-5 validation sample and 0.17 on the month 6–7 test sample, the same ballpark.

**Finding 2 – why: fraud is not "unusual" in BAF.**
The EDA found no single feature that separates the classes strongly (largest KS 0.26) and the fraud signal sits in combinations and in missingness. Fraudulent applications look like ordinary ones, so an outlier detector that never sees labels has little to latch onto. This points to the data rather than to one setting: the 16-setting grid only spans PR-AUC 0.012–0.015.

**Finding 3 – the tuned settings are not meaningfully better than the others.**
Re-fitting the chosen settings with five seeds gives PR-AUC 0.0135–0.0150 on month 5 (mean 0.0141), as wide as the gap between the best and typical grid settings. The seed used everywhere else (30049) is at the top of that range, so quote 0.014–0.015, not 0.0150. Treat the grid winner (`with_categorical`, 300 trees, `max_samples` 256, `max_features` 0.5) as a reasonable default, not as a significant improvement.

**Finding 4 – `contamination` only changes the cut-off.**
PR-AUC is identical for every contamination value (0.0151 on folds A–B), as expected. Mean F1 climbs from 0.017 at 0.005 to 0.031 at 0.05 and then stays flat (0.032 at 0.075–0.10). The raw maximum sits on the edge of the grid (0.10), so the rule used is the start of the plateau: 0.05. At that cut-off the forest flags 5.5% of applications, is right about 1.9% of the time (about twice the base rate) and catches 10% of fraud on the tuning folds.

**Finding 5 – the forest also degrades over time.**
Its ROC-AUC falls from 0.63 (month 3) to 0.60 (month 4) to 0.57 (month 5), and the same training-month cut-off flags 5.8%, 5.1% and 3.6% of applications, so the score distribution is drifting (EDA Finding 12). The team logistic regression holds up better: ROC-AUC 0.895, 0.891, 0.875.

**Finding 6 – the two approaches find different frauds, but the forest adds little.**
At 5% FPR on month 5 (1,411 frauds) the team model catches 699 (49.5%) and the forest 111 (7.9%). Only 74 are caught by both; the forest alone catches 37 (2.6% of frauds) that the team model misses, while 675 (47.8%) are caught by neither. The two scores are essentially uncorrelated over all applications (Spearman −0.04) and only weakly correlated among frauds (0.27). Adding the forest as a second alarm would raise false alarms by up to about 5 percentage points of legitimate applications for 37 extra frauds.

**Finding 7 – selecting features helps, but not enough.**
With only the four numeric features that best separate the classes in the training months (`current_address_months_count`, `credit_risk_score`, `proposed_credit_limit`, `date_of_birth_distinct_emails_4w`), the forest roughly doubles to PR-AUC 0.027 and recall at 5% FPR 17.7% on month 5. That is still about a third of the team model's recall. This diagnostic uses fraud labels to pick the features, so it is no longer a pure anomaly detector. The protected features (age, income, employment status) are excluded throughout, as in the team repo.

*Implication for the report and the product:*
- Keep the supervised classifier as the scoring model. Present the forest as an evaluated alternative that was rejected (A2-28 model selection).
- The forest could still be worth a mention as a "novelty" check for patterns the labels have not seen, but the evidence here does not support putting it in the decision path.
- The selected classifier is now v4 CatBoost. Re-run a properly matched comparison before quoting these historical logistic-regression results as a comparison with that champion.

## 10. Limitations and hand-offs

- The comparison model is the team's logistic regression rebuilt from the repo code (commit 661871e) and re-fitted on each fold's earlier months with the repo's 250,000-row stratified sample. It is not the repo's frozen artifact. If Kha changes the pipeline, re-run Sections 6–7.
- Fold C (train 0–4, validate month 5) is the team repo's own validation split. It is scored here on the whole month; the repo scores a 50,000-row sample, so small differences from its published validation numbers are expected.
- The forest masks all negative `intended_balcon_amount`. The reproduced historical baseline retains its -1-only conversion for reproducibility; the current production pipeline already masks all negative amounts. The historical difference was not tested separately.
- Fold C is one month with 1,411 frauds. The forest's ranking is noisy, so quote ranges, not single values.
- This notebook does not score months 6–7. Other experiments have previously inspected them, so no fresh holdout claim follows from this notebook.
- All tables are in `reports/tables/isoforest_*.csv`; figures are `fig12`–`fig14` (renumber in the order the report cites them).